# Lensed LISA MBHB waveforms
Unlensed MBHB (PyCBC) → LISA TDI A/E/T response → SciRDv1 noise, with a wave-optics lensing
amplification factor F(f) from GWLENS. CPU only. Run in the `lensing_gw` env from the repo root.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd())=='Notebooks' else os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from lisa_mbhb import waveform as W, noise, lensing as L
from lisa_mbhb.constants import YEAR

## 1. Source (source-frame masses + redshift → detector frame)

In [ ]:
m1, m2, dl = W.source_to_detector(5e5, 2.5e5, z=1.0)
params = dict(mass1=m1, mass2=m2, spin1z=0.3, spin2z=0.1, distance=dl, inclination=0.9,
              coa_phase=0.3, psi=0.7, ecl_lon=1.2, ecl_lat=0.4, tc=0.9*YEAR)
print(f"detector frame m1={m1:.2e}, m2={m2:.2e}, dL={dl:.0f} Mpc")

## 2. Lens: point mass (exact F(w,y) from GWLENS)
`mass_lz` is the redshifted lens mass in Msun, `y` the impact parameter.

In [ ]:
lens = L.PointMassLens(mass_lz=1e7, y=0.5)
sig = L.LensedMBHB(params, lens)
print('grid points:', len(sig.f), ' w_max =', lens.w(sig.f[-1]))
print('SNR unlensed = %.0f, lensed = %.0f, mismatch = %.3f' % (sig.snr(False), sig.snr(), sig.mismatch()))

## 3. Amplification factor and lensed waveform

In [ ]:
hp0, hc0, _ = sig.polarizations(lensed=False)
hp, hc, F = sig.polarizations()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].semilogx(sig.f, np.abs(F), label='|F|'); ax[0].semilogx(sig.f, np.unwrap(np.angle(F)), label='phase')
ax[0].set_xlabel('f [Hz]'); ax[0].legend(); ax[0].set_title('amplification factor')
ax[1].loglog(sig.f, 2*sig.f*np.abs(hp0), label='unlensed'); ax[1].loglog(sig.f, 2*sig.f*np.abs(hp), label='lensed')
ax[1].set_xlabel('f [Hz]'); ax[1].set_ylabel(r'$2f|\tilde h_+|$'); ax[1].legend()
plt.tight_layout()

## 4. LISA TDI channels and sensitivity projection

In [ ]:
tdi = sig.tdi(); tdi0 = sig.tdi(lensed=False)
fg = np.logspace(-5, 0, 800)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(fg, np.sqrt(fg*noise.sensitivity_curve(fg)), label='SciRDv1')
ax.loglog(fg, np.sqrt(fg*noise.sensitivity_curve(fg, confusion=True)), '--', label='+ confusion')
ax.loglog(sig.f, 2*sig.f*np.abs(tdi0['A']), alpha=.6, label='A unlensed')
ax.loglog(sig.f, 2*sig.f*np.abs(tdi['A']), alpha=.6, label='A lensed')
ax.set_ylim(1e-21, 1e-15); ax.set_xlabel('f [Hz]'); ax.legend()

## 5. Dependence on impact parameter y

In [ ]:
for y in (0.3, 0.5, 1.0, 2.0):
    s = L.LensedMBHB(params, L.PointMassLens(1e7, y))
    print(f'y={y}: SNR lensed = {s.snr():.0f}, mismatch = {s.mismatch():.3f}')

## 6. Other lens models (NFW, composite)
Use a GWLENS table (`gwlens_interpolate`) with the matching dimensionless frequency:
```python
from gwlens.tools import halo
nfw = halo.NFW(M200=1e10, zl=0.5)
lens = L.LookupTableLens('nfw_table.h5', w_of_f=nfw.Dimensionless_Frequency, y=0.5)
L.LensedMBHB(params, lens).snr()
```
Any object with an `amplification(f)` method works, e.g. a GLoW-based wrapper.

*Note:* `mismatch` here is not maximised over time/phase shifts, so it overstates distinguishability.